In [0]:
# 02_ingest — ADLS raw/ JSONL → Bronze Delta 적재

In [0]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("storage_account", "acdriftadls")
dbutils.widgets.text("raw_base", "abfss://lakehouse@acdriftadls.dfs.core.windows.net/raw/dev")

CATALOG = dbutils.widgets.get("catalog")
STORAGE = dbutils.widgets.get("storage_account")
RAW_BASE_PARAM = dbutils.widgets.get("raw_base")
RAW_BASE = RAW_BASE_PARAM or f"abfss://lakehouse@{STORAGE}.dfs.core.windows.net/raw"

In [0]:
from pyspark.sql import functions as F

SCHEMA_VERSION = "0.1.0"

def _get_run_id():
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().get()
    except Exception:
        return None

def add_meta(df, src_system, src_object, file_path):
    src_cols = df.columns  # 메타 추가 전 원본 컬럼 목록 캡처
    raw_payload = F.to_json(F.struct(*src_cols))
    return (
        df
        .withColumn("_src_system",     F.lit(src_system))
        .withColumn("_src_object",     F.lit(src_object))
        .withColumn("_src_file_path",  F.lit(file_path))
        .withColumn("_ingestion_ts",   F.current_timestamp())
        .withColumn("_run_id",         F.lit(_get_run_id()))
        .withColumn("_batch_id",       F.date_format(F.current_timestamp(), "yyyyMMddHHmmss"))
        .withColumn("_raw_payload",    raw_payload)
        .withColumn("_checksum",       F.md5(raw_payload))
        .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    )

## azure/applications → bronze_azure_applications

In [0]:
path = f"{RAW_BASE}/azure/applications.jsonl"
df = spark.read.json(path)
df = add_meta(df, "entra_id", "applications", path)
df = (df
      .withColumnRenamed("appId", "app_id")
      .withColumnRenamed("displayName", "display_name")
      .withColumn("created_at", F.to_timestamp("createdDateTime"))
      .drop("createdDateTime"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "app_id","display_name","created_at"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_azure_applications"))
print(f"bronze_azure_applications: {df.count()} rows")

## azure/application_owners → bronze_azure_application_owners

In [0]:
path = f"{RAW_BASE}/azure/application_owners.jsonl"
df = spark.read.json(path)
df = add_meta(df, "entra_id", "application_owners", path)
df = (df
      .withColumnRenamed("appId", "app_id")
      .withColumnRenamed("userPrincipalName", "user_principal_name"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "app_id","user_principal_name"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_azure_application_owners"))
print(f"bronze_azure_application_owners: {df.count()} rows")

## azure/password_credentials → bronze_azure_password_credentials

In [0]:
path = f"{RAW_BASE}/azure/password_credentials.jsonl"
df = spark.read.json(path)
df = add_meta(df, "entra_id", "password_credentials", path)
df = (df
      .withColumnRenamed("appId", "app_id")
      .withColumnRenamed("keyId", "key_id")
      .withColumnRenamed("displayName", "display_name")
      .withColumn("end_date_time", F.to_timestamp("endDateTime"))
      .drop("endDateTime"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "app_id","key_id","display_name","end_date_time","status"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_azure_password_credentials"))
print(f"bronze_azure_password_credentials: {df.count()} rows")

## identity/lifecycle_seed → bronze_identity_lifecycle_seed

In [0]:
path = f"{RAW_BASE}/identity/lifecycle_seed.jsonl"
df = spark.read.json(path)
df = add_meta(df, "identity_seed", "lifecycle_seed", path)
df = (df
      .withColumnRenamed("userPrincipalName", "user_principal_name")
      .withColumnRenamed("displayName", "display_name")
      .withColumnRenamed("employmentStatus", "employment_status")
      .withColumn("termination_date", F.to_date("terminationDate"))
      .drop("terminationDate"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "user_principal_name","display_name","employment_status","termination_date"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_identity_lifecycle_seed"))
print(f"bronze_identity_lifecycle_seed: {df.count()} rows")

## assets/assets_seed → bronze_assets_seed

In [0]:
path = f"{RAW_BASE}/assets/assets_seed.jsonl"
df = spark.read.json(path)
df = add_meta(df, "assets_seed", "assets_seed", path)
df = df.withColumnRenamed("assetId", "asset_id")

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "asset_id","name","environment","sensitivity"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_assets_seed"))
print(f"bronze_assets_seed: {df.count()} rows")

## access/access_edges_seed → bronze_access_edges_seed

In [0]:
path = f"{RAW_BASE}/access/access_edges_seed.jsonl"
df = spark.read.json(path)
df = add_meta(df, "access_seed", "access_edges_seed", path)
df = (df
      .withColumnRenamed("principalId", "principal_id")
      .withColumnRenamed("assetId", "asset_id"))

if "lastUsedDateTime" in df.columns:
    df = df.withColumn("last_used_at", F.to_timestamp("lastUsedDateTime"))
elif "last_used_at" in df.columns:
    df = df.withColumn("last_used_at", F.to_timestamp("last_used_at"))
else:
    df = df.withColumn("last_used_at", F.lit(None).cast("timestamp"))

(df.select(
    "_src_system","_src_object","_src_file_path","_ingestion_ts",
    "_run_id","_batch_id","_checksum","_raw_payload","_schema_version",
    "principal_id","asset_id","relationship","permission","status","last_used_at"
).write.format("delta").mode("overwrite")
 .saveAsTable(f"{CATALOG}.bronze.bronze_access_edges_seed"))
print(f"bronze_access_edges_seed: {df.count()} rows")

## Smoke test

In [0]:
bronze_tables = [
    "bronze_azure_applications",
    "bronze_azure_application_owners",
    "bronze_azure_password_credentials",
    "bronze_identity_lifecycle_seed",
    "bronze_assets_seed",
    "bronze_access_edges_seed",
]

summary = ["ingest smoke test passed"]
for table in bronze_tables:
    actual = spark.table(f"{CATALOG}.bronze.{table}").count()
    assert actual > 0, f"{table} has no rows"
    line = f"{table}: {actual} rows"
    print(line)
    summary.append(line)

print("ingest complete")
dbutils.notebook.exit("\n".join(summary))
